# 1. Countdown Puzzle: AST Verifier & Dataset Generator
**HLT 2026 - Reasoning with Verifiable Rewards**

Questo notebook implementa l'ambiente per il gioco del **Countdown**:
1. **Verificatore deterministico basato su AST (Abstract Syntax Tree)** per valutare in modo sicuro e rigoroso le soluzioni generate dagli LLM.
2. **Generatore di problemi solubili** per creare dataset di Train e Test bilanciati.
3. **Suite di Test Unitari** per verificare che non vi siano falle o possibilita di *reward hacking*.

> 💡 **Nota hardware**: Questo notebook **non richiede GPU** e puo girare comodamente su qualsiasi CPU locale o su Google Colab standard.

## 1. Setup & Dipendenze

In [ ]:
import os
import ast
import re
import json
import random
import itertools
from typing import List, Dict, Tuple, Optional, Any

## 2. Il Verificatore Deterministico (AST-based)

In [ ]:
class CountdownVerifier:
    """
    Verificatore deterministico sicuro per il gioco Countdown basato su Python AST.
    """
    ALLOWED_OPERATORS = {
        ast.Add: lambda a, b: a + b,
        ast.Sub: lambda a, b: a - b,
        ast.Mult: lambda a, b: a * b,
        ast.Div: lambda a, b: a / b if b != 0 else None,
        ast.FloorDiv: lambda a, b: a // b if b != 0 else None,
        ast.USub: lambda a: -a,
        ast.UAdd: lambda a: +a
    }

    # Includiamo esplicitamente le classi operatore perche ast.walk le visita come nodi figli
    ALLOWED_NODE_TYPES = (
        ast.Expression,
        ast.BinOp,
        ast.UnaryOp,
        ast.Constant,
        ast.Add,
        ast.Sub,
        ast.Mult,
        ast.Div,
        ast.FloorDiv,
        ast.USub,
        ast.UAdd
    )

    @staticmethod
    def extract_tags(text: str) -> Tuple[Optional[str], Optional[str]]:
        """Estrae thought e answer dai tag XML-like <think> e <answer>."""
        think_match = re.search(r"<think>(.*?)</think>", text, re.DOTALL | re.IGNORECASE)
        thought = think_match.group(1).strip() if think_match else None

        answer_match = re.search(r"<answer>(.*?)</answer>", text, re.DOTALL | re.IGNORECASE)
        answer = answer_match.group(1).strip() if answer_match else None

        if not answer:
            answer_partial = re.search(r"<answer>(.*)", text, re.DOTALL | re.IGNORECASE)
            if answer_partial:
                answer = answer_partial.group(1).strip().split("\n")[0].strip()

        return thought, answer

    @classmethod
    def is_safe_ast(cls, tree: ast.AST) -> bool:
        """Verifica che l'albero contenga SOLO nodi e operatori aritmetici consentiti."""
        for node in ast.walk(tree):
            if not isinstance(node, cls.ALLOWED_NODE_TYPES):
                return False
            if isinstance(node, ast.BinOp) and type(node.op) not in cls.ALLOWED_OPERATORS:
                return False
            if isinstance(node, ast.UnaryOp) and type(node.op) not in cls.ALLOWED_OPERATORS:
                return False
            if isinstance(node, ast.Constant):
                # Esclude stringhe, None e booleani (in Python bool e sottoclasse di int)
                if isinstance(node.value, bool) or not isinstance(node.value, (int, float)):
                    return False
        return True

    @classmethod
    def evaluate_ast(cls, node: ast.AST) -> Optional[float]:
        """Valuta ricorsivamente un nodo AST consentendo solo operazioni matematiche di base."""
        if isinstance(node, ast.Expression):
            return cls.evaluate_ast(node.body)

        elif isinstance(node, ast.Constant) and isinstance(node.value, (int, float)) and not isinstance(node.value, bool):
            return float(node.value)

        elif isinstance(node, ast.UnaryOp):
            op_func = cls.ALLOWED_OPERATORS.get(type(node.op))
            if not op_func:
                return None
            val = cls.evaluate_ast(node.operand)
            return op_func(val) if val is not None else None

        elif isinstance(node, ast.BinOp):
            op_func = cls.ALLOWED_OPERATORS.get(type(node.op))
            if not op_func:
                return None
            left_val = cls.evaluate_ast(node.left)
            right_val = cls.evaluate_ast(node.right)
            if left_val is None or right_val is None:
                return None
            try:
                res = op_func(left_val, right_val)
                return res
            except (ZeroDivisionError, OverflowError):
                return None

        return None

    @classmethod
    def extract_numbers_from_ast(cls, node: ast.AST) -> List[int]:
        """Estrae tutti i numeri utilizzati nell'espressione AST."""
        numbers = []
        for n in ast.walk(node):
            if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)) and not isinstance(n.value, bool):
                if float(n.value).is_integer():
                    numbers.append(int(n.value))
                else:
                    numbers.append(n.value)
        return numbers

    @classmethod
    def verify(cls, available_numbers: List[int], target: int, completion: str) -> Dict[str, Any]:
        """
        Verifica completa del completamento generato dal modello.
        Restituisce un dizionario con lo stato di formattazione, validita e correttezza.
        """
        thought, answer_str = cls.extract_tags(completion)

        result = {
            "has_format": bool(thought is not None and answer_str is not None),
            "thought": thought,
            "answer_str": answer_str,
            "valid_syntax": False,
            "valid_numbers": False,
            "used_numbers": [],
            "computed_value": None,
            "is_correct": False,
            "error_msg": None
        }

        if not answer_str:
            result["error_msg"] = "Tag <answer> non trovato o vuoto"
            return result

        # 1. Parsing AST e Whitelist di sicurezza
        try:
            clean_expr = answer_str.split("=")[0].strip()
            tree = ast.parse(clean_expr, mode='eval')
            if not cls.is_safe_ast(tree):
                result["error_msg"] = "Espressione contiene elementi non consentiti (solo +, -, *, /, parentesi e numeri)"
                return result
            result["valid_syntax"] = True
        except Exception as e:
            result["error_msg"] = f"Errore sintassi AST: {e}"
            return result

        # 2. Controllo dei numeri utilizzati
        used_numbers = cls.extract_numbers_from_ast(tree)
        result["used_numbers"] = used_numbers

        available_copy = list(available_numbers)
        valid_nums = True
        for num in used_numbers:
            if num in available_copy:
                available_copy.remove(num)
            else:
                valid_nums = False
                break

        result["valid_numbers"] = valid_nums and (len(used_numbers) > 0)
        if not result["valid_numbers"]:
            result["error_msg"] = f"Numeri non validi o usati piu volte: {used_numbers} (Disponibili: {available_numbers})"
            return result

        # 3. Calcolo del valore
        computed = cls.evaluate_ast(tree)
        result["computed_value"] = computed

        if computed is None:
            result["error_msg"] = "Valutazione espressione fallita (es. divisione per zero)"
            return result

        # 4. Controllo del target
        if abs(computed - target) < 1e-6:
            result["is_correct"] = True
        else:
            result["error_msg"] = f"Risultato {computed} != target {target}"

        return result

## 3. Unit Test per il Verificatore

In [ ]:
def test_verifier():
    nums = [84, 83, 34, 72]
    target = 39

    # Test 1: Soluzione corretta ben formattata
    out1 = "<think>Risolviamo: (84 + 72) - (83 + 34) = 156 - 117 = 39</think><answer>(84 + 72) - (83 + 34)</answer>"
    res1 = CountdownVerifier.verify(nums, target, out1)
    assert res1["is_correct"] and res1["has_format"], f"Fallito test 1: {res1}"

    # Test 2: Riuso illegale di un numero (84 usato 2 volte)
    out2 = "<think>Proviamo</think><answer>84 + 84 - 72</answer>"
    res2 = CountdownVerifier.verify(nums, target, out2)
    assert not res2["valid_numbers"], "Fallito test 2: accettato riuso non permesso"

    # Test 3: Numero non presente nella lista (es. 10)
    out3 = "<think>Calcolo</think><answer>10 + 29</answer>"
    res3 = CountdownVerifier.verify(nums, target, out3)
    assert not res3["valid_numbers"], "Fallito test 3: accettato numero inesistente"

    # Test 4: Divisione per zero
    out4 = "<think>...</think><answer>84 / (34 - 34)</answer>"
    res4 = CountdownVerifier.verify(nums, target, out4)
    assert not res4["is_correct"], "Fallito test 4: divisione per zero non gestita"

    # Test 5: Sintassi non valida / codice malevolo
    out5 = "<think>...</think><answer>__import__('os').system('ls')</answer>"
    res5 = CountdownVerifier.verify(nums, target, out5)
    assert not res5["valid_syntax"], "Fallito test 5: accettato codice non consentito"

    print("✅ Tutti i 5 Unit Tests del verificatore superati con successo!")

test_verifier()

## 4. Generatore di Puzzle con Solubilita Garantita

In [ ]:
def solve_countdown_all(numbers: List[int]) -> Dict[int, str]:
    """Trova tutti i target interi positivi raggiungibili con i numeri dati."""
    targets = {}

    for length in range(2, len(numbers) + 1):
        for subset in itertools.permutations(numbers, length):
            def eval_tree(items):
                if len(items) == 1:
                    return [(items[0], str(items[0]))]
                results = []
                for split in range(1, len(items)):
                    lefts = eval_tree(items[:split])
                    rights = eval_tree(items[split:])
                    for lv, le in lefts:
                        for rv, re in rights:
                            results.append((lv + rv, f"({le} + {re})"))
                            if lv - rv > 0:
                                results.append((lv - rv, f"({le} - {re})"))
                            results.append((lv * rv, f"({le} * {re})"))
                            if rv > 0 and lv % rv == 0:
                                results.append((lv // rv, f"({le} / {re})"))
                return results

            for v, e in eval_tree(list(subset)):
                if 1 <= v <= 200:
                    targets[v] = e

    return targets

def generate_countdown_dataset(num_samples: int, seed: int = 42) -> List[Dict[str, Any]]:
    """Genera un dataset di puzzle Countdown garantiti solubili."""
    random.seed(seed)
    dataset = []
    seen = set()

    attempts = 0
    while len(dataset) < num_samples and attempts < num_samples * 25:
        attempts += 1
        numbers = [random.randint(1, 100) for _ in range(4)]
        key = tuple(sorted(numbers))
        if key in seen:
            continue

        reachable = solve_countdown_all(numbers)
        valid_targets = [t for t in reachable.keys() if t not in numbers and t >= 10]

        if valid_targets:
            chosen_target = random.choice(valid_targets)
            sample = {
                "id": len(dataset) + 1,
                "numbers": numbers,
                "target": chosen_target,
                "example_solution": reachable[chosen_target],
                "prompt": f"Using the numbers {numbers}, create an equation that equals {chosen_target}. You can use basic arithmetic operations (+, -, *, /) and each number can be used at most once. Show your reasoning in <think>...</think> and place the final equation in <answer>...</answer>."
            }
            dataset.append(sample)
            seen.add(key)

    return dataset

## 5. Generazione & Esportazione Train e Test Set

In [ ]:
os.makedirs("../data", exist_ok=True)

print("Generazione dataset di Test (200 campioni)...")
test_data = generate_countdown_dataset(200, seed=123)

print("Generazione dataset di Train (1000 campioni)...")
train_data = generate_countdown_dataset(1000, seed=456)

with open("../data/test_countdown.jsonl", "w", encoding="utf-8") as f:
    for item in test_data:
        f.write(json.dumps(item) + "\n")

with open("../data/train_countdown.jsonl", "w", encoding="utf-8") as f:
    for item in train_data:
        f.write(json.dumps(item) + "\n")

print(f"Salvati: {len(train_data)} train samples, {len(test_data)} test samples in data/")
print("\nEsempio di problema:")
print(json.dumps(test_data[0], indent=2))